# 7.7 填充的位置會被當線索嗎？


在真實輸入前面補兩個PAD，再接ID1、2、3，最後位置從五個格裡看到的過去與原三字串不同。如果只用因果遮罩，它會允許讀那兩個PAD，哪怕PAD的答案loss已忽略。因此 [7.6](https://birdhackor.github.io/tiny-perceptron-vlm/7.6.html) 的損失規則還不足以保證結果相同，我們還需要遮掉無效key，並保持真實位置的編號。

[3.6](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html) 的因果條件是「不能讀未來」，而key_valid條件是「不能讀填充」。兩者要同時成立，叫邏輯AND。再看 [4.1](https://birdhackor.github.io/tiny-perceptron-vlm/4.1.html)，原始ID1在位置0，左補齊後物理格2仍應使用語義位置0，不應突然換成位置向量2。物理索引是tensor欄位，位置ID則是模型使用的順序約定。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.model import TinyLM, ModelConfig

torch.manual_seed(42)
model = TinyLM(ModelConfig(vocab_size=10, width=8))
base = model(torch.tensor([[1, 2, 3]]))["logits"]
ids = torch.tensor([[0, 0, 1, 2, 3]])
valid = ids != 0
positions = torch.tensor([[0, 0, 0, 1, 2]])
actual = model(ids, valid=valid, positions=positions)["logits"][:, 2:]
print("有效位置最大差", (base - actual).abs().max().item())
assert torch.allclose(base, actual, atol=1e-6)

輸入base是一筆三真token。補齊版ids用0當PAD，valid是 `[False,False,True,True,True]`，positions最後三格仍0、1、2；PAD位置的數字不用於真實對應。模型把valid與causal合成讀取許可，輸出取 `[:,2:]` 只比較真位置。最大差應為0或極小浮點誤差，檢查容許百萬分之一的絕對差。

本例假定ID0專供PAD，真實內容不使用0，所以 `ids!=0` 能判斷有效。換成別的字表要改相同約定，不能無條件把所有0當空格。一般batch工具會直接提供valid，不必從token值猜，後面都應保留它。

最前面PAD查詢可能沒有任何有效key；若對全負無窮分數直接softmax會出NaN，因為無法形成合法比例。專案的注意力工具會把這種空讀取輸出設0，再由valid與答案mask保證它不干擾真實目標。處理空行是為了數值完整，不表示PAD開始有了需學的意思。

若取消valid，真實位置就可能混入PAD向量；若只取消自訂positions，它們則改讀位置2、3、4的向量，另一種變化也會使等價檢查失敗。兩個條件分別解決內容和順序，不能用一個漂亮的三角矩陣代替所有填充規則。

練習先只刪調用裡的 `valid=valid`，保留positions，預測差值通常不再接近0、assert失敗，再執行核對。恢復valid後，再只刪positions重複一次；定位每種差異的來源。比較時固定seed與模型，避免重新隨機初始化把PAD影響混進其他變化。
